# TIDE-GR **v2** on Amazon Reviews 2023 — Video_Games 5-core (Colab T4) — branch `v2`

Runs version 2 without touching v1: code from the `v2` branch, results in a **separate** Drive folder and Sheet.

Dataset (checked 2026-11-24): 811k ratings, 94.6k users, 25.6k items; history = before 2018 (13 blocks),
stream = 21 quarters 2018Q1–2023Q1; ~80% of stream ratings come from returning users, ~12% hit new items.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Resume after a disconnect: run the cells
again (skip the quick check); finished runs are skipped and the unfinished one continues from its checkpoint.


In [ ]:
!nvidia-smi -L

## 1. Settings — the v2 runs folder must be different from the v1 folder

In [ ]:
RUNS_ROOT = "/content/drive/MyDrive/TIDE_GR_v2_runs"   # <- v2 folder (can be the same v2 folder as KuaiRand)
REPO = "https://github.com/S-V-Kartheek/TIDE_GR.git"
BRANCH = "v2"
MAX_HOURS = 11.0
SHEET = "TIDE-GR v2 results"   # <- URL of the v2 Google Sheet (Amazon gets its own amazon_vg_* tabs)
SA_KEY = None
assert "TIDE_GR_runs" != RUNS_ROOT.rstrip("/").split("/")[-1], "use a separate folder for v2"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(RUNS_ROOT, exist_ok=True)
print("runs folder:", RUNS_ROOT, "contents:", os.listdir(RUNS_ROOT))

## 2. Get / update the code
If the repository is private, add a Colab secret named `GITHUB_TOKEN` (key icon on the left) with a GitHub token that can read the repo.

In [ ]:
import os, subprocess
url = REPO
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    if tok:
        url = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists('/content/TIDE_GR'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/TIDE_GR'], check=True)
else:
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'pull', '-q', 'origin', BRANCH], check=True)
%cd /content/TIDE_GR
!git branch --show-current
!git log -1 --oneline
!pip install -q -r requirements.txt


## 3. Quick check (optional, ≈15–30 min, run once)
Downloads the data, builds the stream and the item-text embeddings (cached in the v2 folder), checks metrics + resume.

In [ ]:
!python -m tidegr.quick_check --data data/amazon_vg.yaml --runs_root "$RUNS_ROOT"

## 4. Full v2 plan (resumable — just run this cell again after any interruption)
The run called `tidegr` is TIDE-GR **v2**; `tidegr_v1` is the unchanged v1 model re-run for comparison.

In [ ]:
!python -m tidegr.run_all --plan plans/amazon_vg_v2.yaml --runs_root "$RUNS_ROOT" --max_hours $MAX_HOURS

## 5. Progress

In [ ]:
import json
st = json.load(open(f"{RUNS_ROOT}/amazon_vg/status.json"))
for k, v in st.items():
    print(f"{k:35s} {v.get('state'):8s} {v.get('ndcg@20') or ''}")

## 6. Results → Drive + Google Sheet + paper figures (run any time; re-run after new runs finish)
* Tables: `<RUNS_ROOT>/amazon_vg/results/` (summary.csv, table.md, significance.csv)
* Figures (PDF for LaTeX + PNG): `<RUNS_ROOT>/amazon_vg/results/figures/`
* Google Sheet tabs: `amazon_vg_table`, `amazon_vg_runs`, `amazon_vg_blocks`, `amazon_vg_significance`, `amazon_vg_status`, `log`
The first time, Google asks you to allow access (Sheets + Drive). Each run re-writes the tabs from the files on Drive.

In [ ]:
!python -m tidegr.report --runs_root "$RUNS_ROOT" --dataset amazon_vg

In [ ]:
import sys; sys.path.insert(0, '/content/TIDE_GR')
from tidegr.sheets import sync
sync(RUNS_ROOT, 'amazon_vg', SHEET, sa_key=SA_KEY)

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f'{RUNS_ROOT}/amazon_vg/results/figures/*.png')):
    print(p); display(Image(p, width=520))